# 50 — M4 forward derivatives and complete error ledger

## 判定（2026-07-20 UTC）

**M3 は独立レビュー済みで、M4へ進めます。** M4は5つのsource channelをfixed-basis projection、contraction、regrouping、normalizationへforward伝播し、全error termをprovenance付きDAGへ保存します。

M3のRSVD残差、GPU rounding、basis variation、normalization lower bound、cutoff/rank依存にはdeterministic upper boundがありません。したがってM4の完了状態は `BLOCKED_MATH / NOT_CERTIFIED` です。M5は自動開始しません。


## session時間と人間による再開

- 今回作る新規M4 runだけ: 1時間30分以降は長い項目を開始せず、1時間50分までにfinal save、2時間でreturn。
- 同じrun IDを次回fresh kernelで再開: 5時間以降は長い項目を開始せず、5時間20分までにfinal save、5時間30分でreturn。
- 15分ごと、各work itemの前後、全phase境界でatomic checkpoint。

再開時は下記のrun IDを、実行時に表示された値へ置き換えます。

```bash
export VALIDATED_RG_PERSIST_ROOT=/storage/validated_4d_su2_rg
export VALIDATED_RG_PERSIST_ACK=I_CONFIRM_THIS_PATH_IS_PERSISTENT
export VALIDATED_RG_M3_RUN_ID=M3-20260720T013551Z-ae995e91e861
export VALIDATED_RG_M4_RUN_ID=M4-...
```

fresh kernelでこのnotebookを上から実行します。最新checkpointが破損していれば一つ前へfallbackし、`running` itemは`pending`へ戻ります。


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'AGENTS.md').is_file():
    raise RuntimeError(f'Project rootを特定できません: {PROJECT_ROOT}')
sys.path.insert(0, str(PROJECT_ROOT))

missing = [
    requirement for requirement, module in (
        ('pytest>=8', 'pytest'), ('sympy>=1.12', 'sympy'),
        ('opt_einsum>=3.3', 'opt_einsum'),
    ) if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import pytest
import torch
print('project:', PROJECT_ROOT)
print('pytest:', pytest.__version__)
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('実M4 runにはCUDA GPUが必要です。')
print('GPU:', torch.cuda.get_device_properties(0).name)


In [ ]:
from src.m4_config import M4Config
from src.m4_parent import verify_accepted_m3_parent

M4_CONFIG = M4Config()
M3_EVIDENCE = verify_accepted_m3_parent(PROJECT_ROOT, M4_CONFIG)
print(json.dumps({
    '判定': 'M3受理。M4 forward derivative/error ledgerへ進行可。',
    'parent_run': M4_CONFIG.parent_run_id,
    'parent_hashes': M3_EVIDENCE.hashes,
    'parent_tensor_shapes': {k: list(v.shape) for k, v in M3_EVIDENCE.tensors.items()},
    'screening': M3_EVIDENCE.metrics['screening'],
    'M4_target_status': 'BLOCKED_MATH',
    'certification_status': 'NOT_CERTIFIED',
}, ensure_ascii=False, indent=2))


## M4で伝播するsource class

- temporal link
- spatial link
- electric-like
- magnetic-like / plaquette-like
- low-representation boundary mode

有限差分はforward derivativeの回帰検算だけに使います。誤差DAGはinitial representation tail、basis equivalence、input radius、GPU rounding/backward、RSVD residual、basis variation、omitted channel tail、normalization、tangent、cutoff/rank依存をすべて列挙し、同じleafの二重計上を拒否します。


In [ ]:
test_started = time.monotonic()
cpu = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', '-m', 'not gpu'],
    cwd=PROJECT_ROOT, check=False,
)
if cpu.returncode != 0:
    raise RuntimeError(f'M0–M4 CPU suite failed: exit={cpu.returncode}')
gpu = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q',
     'tests/test_forward_ad.py', 'tests/test_m4_restart.py', '-m', 'gpu'],
    cwd=PROJECT_ROOT, check=False,
)
if gpu.returncode != 0:
    raise RuntimeError(f'M4 required GPU suite failed: exit={gpu.returncode}')

M4_TEST_REPORT = {
    'accepted_m3_parent': 'PASS',
    'm0_m1_m2_m3_regression_cpu_suite': 'PASS',
    'm4_required_cpu_suite': 'PASS',
    'm4_required_gpu_suite': 'PASS',
    'm4_fresh_process_resume': 'PASS',
    'm4_derivative_checkpoint_restore': 'PASS',
    'elapsed_s': time.monotonic() - test_started,
}
print(json.dumps(M4_TEST_REPORT, indent=2))


In [ ]:
from src.m4_orchestrator import create_or_resume_m4
from src.runtime import environment_info, validate_persistent_root

PERSIST_ROOT = validate_persistent_root()
print(json.dumps(environment_info(), ensure_ascii=False, indent=2))
m4_orchestrator = create_or_resume_m4(
    PERSIST_ROOT, M4_CONFIG, PROJECT_ROOT,
    run_id=os.environ.get('VALIDATED_RG_M4_RUN_ID'),
    test_report=M4_TEST_REPORT,
)
print('再開用: export VALIDATED_RG_M4_RUN_ID=' + m4_orchestrator.state.run_id)
print('session policy:', m4_orchestrator.session_policy)
assert m4_orchestrator.state.certification_status == 'NOT_CERTIFIED'
M4_RESULT = m4_orchestrator.run_until_checkpoint()
assert M4_RESULT['certification_status'] == 'NOT_CERTIFIED'
assert M4_RESULT['milestone_status'] == 'BLOCKED_MATH'
M4_RESULT


In [ ]:
from src.common import read_json

loaded = m4_orchestrator.checkpoints.load_latest(restore_rng=False)
if loaded is None:
    raise RuntimeError('有効なM4 checkpointがありません。')
inspection = {
    'run_id': loaded.state.run_id,
    'phase': loaded.state.phase,
    'checkpoint': str(loaded.path),
    'checkpoint_index': loaded.state.checkpoint_index,
    'done': sum(item.status == 'done' for item in loaded.queue.items.values()),
    'pending': sum(item.status == 'pending' for item in loaded.queue.items.values()),
    'derivative_tensors': sorted(loaded.tensors),
    'skipped_invalid': list(loaded.skipped_invalid),
    'certification_status': loaded.state.certification_status,
}
print(json.dumps(inspection, ensure_ascii=False, indent=2))

report_path = m4_orchestrator.run_root / 'reports/M4_report.json'
if report_path.is_file():
    report = read_json(report_path)
    ledger = report['results']['M4_ERROR_LEDGER']['result']
    print(json.dumps({
        '判定': 'M4 derivative実装完了。ただしBLOCKED_MATHでM5へ進行不可。',
        'acceptance_gates': report['acceptance_gates'],
        'missing_bound_terms': report['missing_bound_terms'],
        'partial_output_radii': ledger['aggregates'],
        'checkpoint': report['checkpoint'],
        'memory': report['memory'],
        'gpu_memory': report['gpu_memory'],
        'certification_status': report['certification_status'],
    }, ensure_ascii=False, indent=2))
else:
    print('M4未完了です。reports/next_session_plan.mdに従って同じrun IDを再開してください。')
